In [2]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split

# -----------------------------
# CONFIG
# -----------------------------
BATCH_SIZE = 16
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

print(device)

# -----------------------------
# LOAD DATA
# -----------------------------
X_lig = np.load("data/processed_features/X_lig.npy", mmap_mode='r')
X_prot = np.load("data/processed_features/X_prot.npy", mmap_mode='r')
y = np.load("data/processed_features/y.npy", mmap_mode='r')

print("Ligand shape:", X_lig.shape)
print("Protein shape:", X_prot.shape)

# -----------------------------
# SPLIT
# -----------------------------
Xl_train, Xl_test, Xp_train, Xp_test, y_train, y_test = train_test_split(
    X_lig, X_prot, y, test_size=0.2, random_state=42
)

mps
Ligand shape: (50000, 1024)
Protein shape: (50000, 1024)


In [3]:
Xl_train
X_prot

memmap([[ 0.01121809,  0.14830887, -0.0471754 , ..., -0.03238064,
         -0.08628611, -0.00524454],
        [ 0.01121809,  0.14830887, -0.0471754 , ..., -0.03238064,
         -0.08628611, -0.00524454],
        [ 0.01121809,  0.14830887, -0.0471754 , ..., -0.03238064,
         -0.08628611, -0.00524454],
        ...,
        [ 0.09514105,  0.09496618,  0.00456134, ..., -0.12394017,
         -0.10022811, -0.03949062],
        [ 0.09514105,  0.09496618,  0.00456134, ..., -0.12394017,
         -0.10022811, -0.03949062],
        [ 0.09514105,  0.09496618,  0.00456134, ..., -0.12394017,
         -0.10022811, -0.03949062]], shape=(50000, 1024), dtype=float32)

In [12]:
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, roc_auc_score


# -----------------------------
# CONFIG
# -----------------------------
FP_SIZE = 1024
MAX_LEN = 300
BATCH_SIZE = 16
EPOCHS = 50

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

# -----------------------------
# DATASET
# -----------------------------
class DTIDataset(Dataset):
    def __init__(self, lig, prot, y):
        self.lig = lig
        self.prot = prot
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.lig[idx], dtype=torch.float32),
            torch.tensor(self.prot[idx], dtype=torch.long),
            torch.tensor(self.y[idx], dtype=torch.float32),
        )


train_loader = DataLoader(DTIDataset(Xl_train, Xp_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True)

test_loader = DataLoader(DTIDataset(Xl_test, Xp_test, y_test),
                         batch_size=BATCH_SIZE)


# -----------------------------
# MODEL
# -----------------------------
class DTIModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.ligand_net = nn.Sequential(
            nn.Linear(FP_SIZE, 256),
            nn.ReLU(),
            nn.Linear(256, 64)
        )

        self.embedding = nn.Embedding(21, 64, padding_idx=0)

        self.fc = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, ligand, protein):
        l = self.ligand_net(ligand)

        p = self.embedding(protein)
        p = p.mean(dim=1)

        x = torch.cat([l, p], dim=1)
        return self.fc(x).squeeze()


model = DTIModel().to(device)


# -----------------------------
# LOSS (IMPORTANT)
# -----------------------------
pos = torch.tensor(np.sum(y_train == 1))
neg = torch.tensor(np.sum(y_train == 0))

pos_weight = torch.tensor([neg / pos]).to(device)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


# -----------------------------
# TRAIN
# -----------------------------
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for ligand, protein, label in train_loader:
        ligand = ligand.to(device)
        protein = protein.to(device)
        label = label.to(device)

        pred = model(ligand, protein)

        loss = loss_fn(pred, label)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")


# -----------------------------
# EVALUATION
# -----------------------------
model.eval()
preds, probs, true = [], [], []

with torch.no_grad():
    for ligand, protein, label in test_loader:
        ligand = ligand.to(device)
        protein = protein.to(device)

        pred = model(ligand, protein)
        prob = torch.sigmoid(pred)

        preds.extend((prob > 0.5).cpu().numpy())
        probs.extend(prob.cpu().numpy())
        true.extend(label.numpy())

acc = accuracy_score(true, preds)
auc = roc_auc_score(true, probs)

print("Accuracy:", acc)
print("ROC-AUC:", auc)

Using device: mps
Epoch 1, Loss: 1757.8379
Epoch 2, Loss: 1482.0241
Epoch 3, Loss: 1332.4024
Epoch 4, Loss: 1230.7040
Epoch 5, Loss: 1154.7834
Epoch 6, Loss: 1100.3243
Epoch 7, Loss: 1053.2266
Epoch 8, Loss: 1014.8306
Epoch 9, Loss: 989.5873
Epoch 10, Loss: 964.7691
Epoch 11, Loss: 935.6413
Epoch 12, Loss: 924.3680
Epoch 13, Loss: 905.0587
Epoch 14, Loss: 884.5232
Epoch 15, Loss: 877.8617
Epoch 16, Loss: 868.0729
Epoch 17, Loss: 854.9078
Epoch 18, Loss: 849.9692
Epoch 19, Loss: 830.5907
Epoch 20, Loss: 832.8316
Epoch 21, Loss: 821.5898
Epoch 22, Loss: 821.1736
Epoch 23, Loss: 802.8391
Epoch 24, Loss: 805.4049
Epoch 25, Loss: 795.4838
Epoch 26, Loss: 785.9633
Epoch 27, Loss: 787.8769
Epoch 28, Loss: 775.6875
Epoch 29, Loss: 770.6880
Epoch 30, Loss: 773.3991
Epoch 31, Loss: 759.3089
Epoch 32, Loss: 759.6188
Epoch 33, Loss: 764.7256
Epoch 34, Loss: 753.2191
Epoch 35, Loss: 756.8284
Epoch 36, Loss: 744.3077
Epoch 37, Loss: 750.5708
Epoch 38, Loss: 741.7199
Epoch 39, Loss: 744.9845
Epoch 40

In [4]:
def prepare_features(Xl, Xp):
    Xp_mean = np.mean(Xp, axis=1)

    Xp_mean = Xp_mean.reshape(-1, 1)

    return np.concatenate([Xl, Xp_mean], axis=1)

Xl_train_ml = prepare_features(Xl_train, Xp_train)
Xl_test_ml = prepare_features(Xl_test, Xp_test)

y_train_ml = y_train
y_test_ml = y_test

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(probability=True, kernel='rbf', class_weight='balanced')
svm_model.fit(Xl_train_ml, y_train_ml)

svm_probs = svm_model.predict_proba(Xl_test_ml)[:, 1]
svm_preds = (svm_probs > 0.5).astype(int)

svm_acc = accuracy_score(y_test_ml, svm_preds)
svm_auc = roc_auc_score(y_test_ml, svm_probs)

print("SVM Accuracy:", svm_acc)
print("SVM ROC-AUC:", svm_auc)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(Xl_train_ml, y_train_ml)  

rf_probs = rf_model.predict_proba(Xl_test_ml)[:, 1]
rf_preds = (rf_probs > 0.5).astype(int)

rf_acc = accuracy_score(y_test_ml, rf_preds)
rf_auc = roc_auc_score(y_test_ml, rf_probs)

print("Random Forest Accuracy:", rf_acc)
print("Random Forest ROC-AUC:", rf_auc)